This is a program which is in testing phase. We will use a linear regression ML model to give an SOC number for the flights.

In [1]:
# Set the testing flights and training flight
train_flight_one_id = 4620
train_flight_two_id = 4929
test_flight_id = 4940

In [2]:
# Import the querying module
from flight_querying import query_flights
import pandas as pd
from storage import execute

# Set up and retrieve the data from the database.
db_connect = query_flights()

In [3]:
train_one_data = db_connect.connect_flight_for_ml(train_flight_one_id)
train_two_data = db_connect.connect_flight_for_ml(train_flight_two_id)

# Set up train data
train_x = pd.concat([train_one_data, train_two_data], axis=0)
train_y = train_x["soc"].to_numpy()
numeric_train_x = train_x.drop(columns=["operations", "soc"])

# Set up test data
test_x = db_connect.connect_flight_for_ml(test_flight_id)
test_y = test_x["soc"].to_numpy()
numeric_test_x = test_x.drop(columns=["operations", "soc"])

In [4]:
train_x

,time,operations,soc,cell_temperature,motor_rpm,motor_power,motor_temperature,indicated_air_speed,pressure_altitude,ground_speed,outside_air_temperature,inverter_temperature
0,0.000000,NA,0,0,0,0,0.000000,0.0,0.000000,0.0,0.0,0.000000
1,0.000000,NA,0,0,0,0,0.000000,0.0,0.000000,0.0,0.0,0.000000
2,0.035273,NA,100,0,0,0,0.000000,0.0,0.000000,0.0,0.0,0.000000
3,0.037968,NA,100,0,0,0,0.000000,0.0,0.000000,0.0,0.0,0.000000
4,0.038609,NA,100,0,0,0,0.000000,0.0,0.000000,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
21744,36.446423,NA,56,17,0,0,36.045551,0.0,302.672821,0.0,13.5,31.879438
21745,36.449079,NA,56,17,0,0,36.045551,0.0,302.672821,0.0,13.5,31.879438
21746,36.449757,NA,56,17,0,0,36.045551,0.0,302.672821,0.0,13.5,31.879438
21747,36.452369,NA,56,17,0,0,36.045551,0.0,302.672821,0.0,13.5,31.879438


One-Hot-Encoding of the Operations columns

In [5]:
# ONE-HOT ENCODE
# https://stackabuse.com/one-hot-encoding-in-python-with-pandas-and-scikit-learn/
def one_hot(df, col, pre):
  encoded = pd.get_dummies(df[col], prefix=pre)
  for column in encoded:
    encoded = encoded.rename(columns={column: col + "_" + column})
  encoded['time'] = df['time']
  return encoded

In [6]:
# Encode Train data
train_encoded = one_hot(train_x, "operations", 'is')
numeric_train_x = pd.merge(numeric_train_x, train_encoded, on='time', how='inner')

# Encode Test data
test_encoded = one_hot(test_x, "operations", 'is')
numeric_test_x = pd.merge(numeric_test_x, test_encoded, on='time', how='inner')

In [7]:
train_encoded

,operations_is_NA,operations_is_climb,operations_is_descent,operations_is_landing,operations_is_takeoff,time
0,True,False,False,False,False,0.000000
1,True,False,False,False,False,0.000000
2,True,False,False,False,False,0.035273
3,True,False,False,False,False,0.037968
4,True,False,False,False,False,0.038609
...,...,...,...,...,...,...
21744,True,False,False,False,False,36.446423
21745,True,False,False,False,False,36.449079
21746,True,False,False,False,False,36.449757
21747,True,False,False,False,False,36.452369


In [8]:
len(numeric_train_x)

53058

In [9]:
len(train_y)

53056

Machine Learning Model Implementation

In [11]:
# import sklearn
from sklearn import preprocessing, svm
from sklearn.linear_model import LinearRegression

# Set model
regression_model = LinearRegression()

# Fit model
regression_model.fit(numeric_train_x[0:53056], train_y)

LinearRegression()

In [14]:
len(numeric_test_x)

26646

In [15]:
len(test_y)

26644

In [16]:
# print model score
print(regression_model.score(numeric_test_x[0:26644], test_y))

0.9512218686138464


In [17]:
len(regression_model.coef_)

15

In [20]:
coeff = pd.DataFrame(list(zip(regression_model.feature_names_in_, regression_model.coef_)), columns = ['Feature', 'Weight'])

In [21]:
coeff.sort_values('Weight')

,Feature,Weight
13,operations_is_landing,-2.959080
10,operations_is_NA,-2.409125
0,time,-1.675925
1,cell_temperature,-0.330616
3,motor_power,-0.106953
5,indicated_air_speed,-0.066785
6,pressure_altitude,-0.018462
7,ground_speed,-0.001818
2,motor_rpm,0.003492
4,motor_temperature,0.229391
